# 📊 Analyse Statistique du Corpus Darija Marocaine
### Classification automatique de textes — Exploration du dataset

---
> **Instructions** : Modifiez la variable `DATASET_PATH` dans la cellule suivante pour pointer vers votre fichier CSV.
> Ensuite : `Kernel → Restart & Run All`

## ⚙️ 0. Installation et Configuration

In [ ]:
# Installation des librairies nécessaires
import subprocess, sys

packages = [
    'pandas', 'numpy', 'plotly', 'matplotlib', 'seaborn',
    'wordcloud', 'arabic-reshaper', 'python-bidi',
    'scikit-learn', 'ipywidgets'
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], 
                   capture_output=True)

print('✅ Packages installés avec succès')

KeyboardInterrupt: 

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from wordcloud import WordCloud
import re
from collections import Counter
import warnings
import ipywidgets as widgets
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 🔧 CONFIGURATION — MODIFIEZ ICI
# ─────────────────────────────────────────────
DATASET_PATH = 'scraper_hajar/data/merged/dataset_merged.csv'   # ← chemin vers votre CSV
THEME_COLOR  = '#003366'             # couleur principale
# ─────────────────────────────────────────────

# Thème Plotly personnalisé
PALETTE = px.colors.qualitative.Set2
PALETTE_RICH = px.colors.qualitative.Plotly

print('✅ Librairies chargées')
print(f'📂 Dataset ciblé : {DATASET_PATH}')

✅ Librairies chargées
📂 Dataset ciblé : scrapping_hajar\darija_dataset\data\merged_2\dataset_merged.csv


---
## 📂 1. Chargement et Aperçu du Dataset

In [3]:
# Chargement
df = pd.read_csv(DATASET_PATH, encoding='utf-8')

# Nettoyage de base
df = df.dropna(subset=['text'])
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'] != '']

# Colonnes utiles
if 'domain' not in df.columns and 'category' in df.columns:
    df['domain'] = df['category']
if 'channel_name' not in df.columns and 'channel' in df.columns:
    df['channel_name'] = df['channel']
if 'label' not in df.columns:
    df['label'] = df.get('domain', 'unknown')
if 'source_type' not in df.columns:
    df['source_type'] = 'youtube_comment'

# Longueur des textes
df['text_length']     = df['text'].str.len()
df['word_count']      = df['text'].str.split().str.len()

# Détection du système d'écriture
def detect_script(text):
    has_arabic = bool(re.search(r'[\u0600-\u06FF]', text))
    has_latin  = bool(re.search(r'[a-zA-Z]', text))
    has_digit_arabizi = bool(re.search(r'[2379]', text))
    if has_arabic and has_latin:
        return 'Mixte'
    elif has_arabic:
        return 'Arabe'
    elif has_latin and has_digit_arabizi:
        return 'Arabizi'
    elif has_latin:
        return 'Latin'
    else:
        return 'Autre'

df['script'] = df['text'].apply(detect_script)

print(f'✅ Dataset chargé : {len(df):,} textes')
print(f'   Colonnes       : {list(df.columns)}')
print(f'   Domaines       : {df["domain"].nunique() if "domain" in df.columns else "N/A"}')
df.head(3)

✅ Dataset chargé : 175,857 textes
   Colonnes       : ['id', 'text', 'source_type', 'domain', 'channel_name', 'published_at', 'label', 'language', 'text_length', 'word_count', 'script']
   Domaines       : 8


,id,text,source_type,domain,channel_name,published_at,label,language,text_length,word_count,script
0,darija_000000,شكرا بزاف بارك الله فيك شرح ممتاز❤,youtube_comment,business,Abdo Nobil,1757462400,business,darija,34,7,Arabe
1,darija_000001,Tooop tbarkallah 3lik 😊,youtube_comment,business,Abdo Nobil,1741564800,business,darija,23,4,Arabizi
2,darija_000002,جزاك الله خيرا,youtube_comment,business,Abdo Nobil,1741564800,business,darija,15,3,Arabe


In [4]:
# ── Carte de synthèse interactive ──────────────────────────────
total     = len(df)
n_domains = df['domain'].nunique() if 'domain' in df.columns else 'N/A'
n_channels= df['channel_name'].nunique() if 'channel_name' in df.columns else 'N/A'
avg_len   = df['text_length'].mean()
n_comments= (df['source_type'] == 'youtube_comment').sum() if 'source_type' in df.columns else 0
n_transcr = (df['source_type'] == 'youtube_transcript').sum() if 'source_type' in df.columns else 0

fig = go.Figure()

metrics = [
    ('📝 Total textes',      f'{total:,}',       '#003366'),
    ('🗂️ Domaines',         str(n_domains),      '#0066B3'),
    ('📺 Chaînes',           str(n_channels),     '#0099CC'),
    ('💬 Commentaires',      f'{n_comments:,}',   '#00AAAA'),
    ('🎙️ Transcriptions',   f'{n_transcr:,}',    '#00BB77'),
    ('📏 Long. moy.',        f'{avg_len:.0f} car',  '#66BB33'),
]

for i, (label, val, color) in enumerate(metrics):
    fig.add_trace(go.Indicator(
        mode='number',
        value=None,
        title={'text': f'<b>{label}</b><br><span style="font-size:1.6em;color:{color}">{val}</span>',
               'font': {'size': 14}},
        domain={'row': 0, 'column': i}
    ))

fig.update_layout(
    grid={'rows': 1, 'columns': len(metrics)},
    height=160,
    margin=dict(t=20, b=10, l=10, r=10),
    paper_bgcolor='#f8f9fa',
    title={'text': '📊 Tableau de bord — Corpus Darija Marocaine',
           'font': {'size': 16, 'color': THEME_COLOR}, 'x': 0.5}
)
fig.show()

---
## 🗂️ 2. Distribution par Domaine

In [5]:
domain_counts = df['domain'].value_counts().reset_index()
domain_counts.columns = ['Domaine', 'Nombre']
domain_counts['Pourcentage'] = (domain_counts['Nombre'] / total * 100).round(1)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Répartition par domaine (barres)', 'Répartition par domaine (camembert)'),
    specs=[[{'type': 'bar'}, {'type': 'pie'}]]
)

# Barres
fig.add_trace(
    go.Bar(
        x=domain_counts['Domaine'],
        y=domain_counts['Nombre'],
        text=domain_counts.apply(lambda r: f"{r['Nombre']:,}<br>({r['Pourcentage']}%)", axis=1),
        textposition='outside',
        marker=dict(color=domain_counts['Nombre'],
                    colorscale='Blues', showscale=False),
        hovertemplate='<b>%{x}</b><br>%{y:,} textes<extra></extra>'
    ), row=1, col=1
)

# Camembert
fig.add_trace(
    go.Pie(
        labels=domain_counts['Domaine'],
        values=domain_counts['Nombre'],
        hole=0.4,
        textinfo='label+percent',
        hovertemplate='<b>%{label}</b><br>%{value:,} textes<br>%{percent}<extra></extra>'
    ), row=1, col=2
)

fig.update_layout(
    title='🗂️ Distribution des textes par domaine thématique',
    height=480,
    showlegend=False,
    font=dict(family='Arial', size=12),
    paper_bgcolor='white'
)
fig.show()

---
## 📺 3. Distribution par Chaîne YouTube

In [6]:
if 'channel_name' in df.columns:
    ch_counts = df.groupby(['channel_name', 'domain']).size().reset_index(name='count')
    ch_total  = ch_counts.groupby('channel_name')['count'].sum().sort_values(ascending=True)
    ch_counts = ch_counts[ch_counts['channel_name'].isin(ch_total.index)]

    fig = px.bar(
        ch_counts,
        x='count',
        y='channel_name',
        color='domain',
        orientation='h',
        title='📺 Nombre de textes par chaîne YouTube (coloré par domaine)',
        labels={'count': 'Nombre de textes', 'channel_name': 'Chaîne', 'domain': 'Domaine'},
        color_discrete_sequence=PALETTE_RICH,
        height=max(500, len(ch_total) * 22)
    )
    fig.update_layout(
        yaxis={'categoryorder': 'total ascending'},
        paper_bgcolor='white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    fig.update_traces(
        hovertemplate='<b>%{y}</b><br>%{x:,} textes<extra></extra>'
    )
    fig.show()
else:
    print('⚠️ Colonne channel_name non trouvée')

---
## 📏 4. Analyse de la Longueur des Textes

In [7]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Distribution longueur (caractères)',
        'Distribution nombre de mots',
        'Longueur moyenne par domaine',
        'Boîtes à moustaches par domaine'
    )
)

# Histogramme longueur
fig.add_trace(
    go.Histogram(
        x=df['text_length'].clip(upper=df['text_length'].quantile(0.99)),
        nbinsx=60,
        marker_color='#0066B3',
        opacity=0.75,
        name='Longueur (chars)',
        hovertemplate='%{x} caractères<br>%{y} textes<extra></extra>'
    ), row=1, col=1
)

# Histogramme mots
fig.add_trace(
    go.Histogram(
        x=df['word_count'].clip(upper=df['word_count'].quantile(0.99)),
        nbinsx=50,
        marker_color='#00AAAA',
        opacity=0.75,
        name='Nb mots',
        hovertemplate='%{x} mots<br>%{y} textes<extra></extra>'
    ), row=1, col=2
)

# Longueur moy par domaine
avg_by_domain = df.groupby('domain')['text_length'].mean().sort_values(ascending=False)
fig.add_trace(
    go.Bar(
        x=avg_by_domain.index,
        y=avg_by_domain.values.round(1),
        marker_color='#003366',
        text=avg_by_domain.values.round(0).astype(int),
        textposition='outside',
        name='Moy. longueur',
        hovertemplate='<b>%{x}</b><br>Moy: %{y:.0f} chars<extra></extra>'
    ), row=2, col=1
)

# Boxplots par domaine
for i, domain in enumerate(df['domain'].unique()):
    subset = df[df['domain'] == domain]['text_length'].clip(upper=500)
    fig.add_trace(
        go.Box(
            y=subset,
            name=domain,
            marker_color=PALETTE[i % len(PALETTE)],
            showlegend=False,
            hovertemplate='<b>%{x}</b><br>%{y} chars<extra></extra>'
        ), row=2, col=2
    )

fig.update_layout(
    title='📏 Analyse de la longueur des textes',
    height=700,
    showlegend=False,
    paper_bgcolor='white'
)
fig.show()

---
## 🔤 5. Analyse des Systèmes d'Écriture (Arabe / Latin / Arabizi)

In [8]:
script_counts = df['script'].value_counts().reset_index()
script_counts.columns = ['Script', 'Nombre']

# Crosstab domaine × script
cross = pd.crosstab(df['domain'], df['script'], normalize='index') * 100

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Répartition globale des scripts',
        'Scripts par domaine (%)'
    ),
    specs=[[{'type': 'pie'}, {'type': 'bar'}]]
)

colors_script = ['#003366', '#0099CC', '#00BB77', '#FFAA00', '#CC3333']

fig.add_trace(
    go.Pie(
        labels=script_counts['Script'],
        values=script_counts['Nombre'],
        hole=0.5,
        textinfo='label+percent',
        marker=dict(colors=colors_script),
        hovertemplate='<b>%{label}</b><br>%{value:,} textes (%{percent})<extra></extra>'
    ), row=1, col=1
)

for i, script in enumerate(cross.columns):
    fig.add_trace(
        go.Bar(
            name=script,
            x=cross.index,
            y=cross[script].round(1),
            marker_color=colors_script[i % len(colors_script)],
            hovertemplate='<b>%{x}</b><br>' + script + ': %{y:.1f}%<extra></extra>'
        ), row=1, col=2
    )

fig.update_layout(
    title='🔤 Analyse des systèmes d\'écriture (Arabe / Latin / Arabizi / Mixte)',
    barmode='stack',
    height=450,
    paper_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=-0.25)
)
fig.show()

---
## 📡 6. Distribution par Type de Source

In [9]:
if 'source_type' in df.columns:
    src_by_domain = df.groupby(['domain', 'source_type']).size().reset_index(name='count')

    fig = px.bar(
        src_by_domain,
        x='domain',
        y='count',
        color='source_type',
        barmode='group',
        title='📡 Commentaires vs Transcriptions par domaine',
        labels={'count': 'Nombre de textes', 'domain': 'Domaine', 'source_type': 'Type de source'},
        color_discrete_map={
            'youtube_comment':    '#003366',
            'youtube_transcript': '#00AAAA'
        },
        text_auto=True,
        height=420
    )
    fig.update_traces(
        texttemplate='%{y:,}',
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>%{y:,} textes<extra></extra>'
    )
    fig.update_layout(
        paper_bgcolor='white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02)
    )
    fig.show()

---
## ☁️ 7. Nuages de Mots par Domaine (Interactif)

In [10]:
def make_wordcloud(domain_name, max_words=120):
    texts = df[df['domain'] == domain_name]['text'].dropna()
    full_text = ' '.join(texts.astype(str).tolist())

    # Supprimer URLs et caractères spéciaux
    full_text = re.sub(r'http\S+', '', full_text)
    full_text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', full_text)

    if not full_text.strip():
        print(f'⚠️ Pas assez de texte pour {domain_name}')
        return

    # Couleurs par domaine
    colormap_map = {
        'religion': 'YlOrRd', 'food': 'Greens', 'entertainment': 'Purples',
        'miscill': 'Greys', 'medical': 'Blues', 'business': 'YlGnBu',
        'sports': 'Oranges', 'politics': 'RdPu'
    }
    cmap = colormap_map.get(domain_name.lower(), 'viridis')

    wc = WordCloud(
        width=900, height=400,
        background_color='white',
        max_words=max_words,
        colormap=cmap,
        prefer_horizontal=0.7,
        collocations=False
    ).generate(full_text)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'☁️ Nuage de mots — {domain_name.upper()}  ({len(texts):,} textes)',
                 fontsize=14, fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()

# Widget interactif
domain_widget = widgets.Dropdown(
    options=sorted(df['domain'].unique()),
    description='Domaine :',
    style={'description_width': 'initial'}
)
words_widget = widgets.IntSlider(
    value=100, min=20, max=200, step=10,
    description='Nb mots :',
    style={'description_width': 'initial'}
)
btn = widgets.Button(description='🔄 Générer', button_style='primary')
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output(wait=True)
        make_wordcloud(domain_widget.value, words_widget.value)

btn.on_click(on_click)
display(widgets.HBox([domain_widget, words_widget, btn]))
display(output)

# Affichage initial
with output:
    make_wordcloud(df['domain'].value_counts().index[0])

Output()

---
## 🔠 8. Mots les Plus Fréquents par Domaine

In [11]:
# Mots fréquents — top 15 par domaine
STOPWORDS_DARIJA = {
    'في', 'من', 'على', 'إلى', 'هذا', 'هذه', 'أن', 'كان', 'مع', 'عن',
    'و', 'ف', 'ب', 'ل', 'يا', 'لا', 'ما', 'هي', 'هو', 'نا', 'كي',
    'li', 'fi', 'wa', 'mn', 'la', 'ma', 'had', 'ana', 'nta', 'wla',
    'the', 'de', 'je', 'le', 'la', 'les', 'et', 'en', 'un', 'une'
}

def top_words(domain, n=15):
    texts = df[df['domain'] == domain]['text'].dropna().astype(str)
    words = []
    for t in texts:
        t = re.sub(r'http\S+', '', t)
        t = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', t)
        words.extend([w.lower() for w in t.split()
                      if len(w) > 2 and w.lower() not in STOPWORDS_DARIJA])
    return Counter(words).most_common(n)

domains = sorted(df['domain'].unique())
ncols = 2
nrows = (len(domains) + 1) // ncols

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[f'Top 15 mots — {d}' for d in domains],
    vertical_spacing=0.08,
    horizontal_spacing=0.12
)

for idx, domain in enumerate(domains):
    row = idx // ncols + 1
    col = idx % ncols + 1
    words_freq = top_words(domain)
    if not words_freq:
        continue
    words_list, freqs = zip(*words_freq)

    fig.add_trace(
        go.Bar(
            x=list(freqs),
            y=list(words_list),
            orientation='h',
            marker_color=PALETTE[idx % len(PALETTE)],
            showlegend=False,
            hovertemplate='<b>%{y}</b> : %{x:,} occurrences<extra></extra>'
        ), row=row, col=col
    )

fig.update_layout(
    title='🔠 Top 15 mots les plus fréquents par domaine',
    height=300 * nrows,
    paper_bgcolor='white',
    showlegend=False
)
fig.update_yaxes(autorange='reversed')
fig.show()

---
## 🔍 9. Détection du Code-Switching

In [12]:
def detect_languages(text):
    has_arabic  = bool(re.search(r'[\u0600-\u06FF]', text))
    has_latin   = bool(re.search(r'[a-zA-Z]{2,}', text))
    has_french  = bool(re.search(
        r'\b(je|tu|il|nous|vous|ils|est|sont|avec|pour|dans|sur|par|les|des|une|qui)\b',
        text.lower()
    ))
    has_arabic_nums = bool(re.search(r'(?<![0-9])[2379](?![0-9])', text))

    langs = []
    if has_arabic:      langs.append('Arabe')
    if has_french:      langs.append('Français')
    elif has_latin:     langs.append('Latin/Darija')
    if has_arabic_nums: langs.append('Arabizi')

    if len(langs) == 0: return 'Indéterminé'
    if len(langs) == 1: return langs[0]
    return ' + '.join(langs[:2])

df['lang_mix'] = df['text'].apply(detect_languages)

lang_counts = df['lang_mix'].value_counts().head(10).reset_index()
lang_counts.columns = ['Combinaison', 'Nombre']
lang_counts['%'] = (lang_counts['Nombre'] / total * 100).round(1)

# Crosstab code-switching × domaine
cs_domain = pd.crosstab(df['domain'], df['lang_mix'].isin(
    df['lang_mix'].value_counts().head(4).index
).map({True: 'Code-switching', False: 'Mono-langue'}))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Distribution des combinaisons linguistiques', 'Code-switching par domaine'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}]]
)

fig.add_trace(
    go.Bar(
        x=lang_counts['Nombre'],
        y=lang_counts['Combinaison'],
        orientation='h',
        marker_color='#003366',
        text=lang_counts['%'].astype(str) + '%',
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>%{x:,} textes<extra></extra>'
    ), row=1, col=1
)

for i, col_name in enumerate(cs_domain.columns):
    fig.add_trace(
        go.Bar(
            name=str(col_name),
            x=cs_domain.index,
            y=cs_domain[col_name],
            marker_color=['#003366', '#00AAAA'][i],
            hovertemplate='<b>%{x}</b><br>%{y:,}<extra></extra>'
        ), row=1, col=2
    )

fig.update_layout(
    title='🔍 Analyse du Code-Switching dans le corpus',
    height=430,
    barmode='stack',
    paper_bgcolor='white',
    yaxis=dict(autorange='reversed'),
    legend=dict(orientation='h', yanchor='bottom', y=-0.25)
)
fig.show()

---
## 🔥 10. Heatmap — Longueur moyenne × Domaine × Source

In [13]:
heat_data = df.groupby(['domain', 'source_type'])['text_length'].mean().round(0).unstack(fill_value=0)

fig = go.Figure(go.Heatmap(
    z=heat_data.values,
    x=[c.replace('youtube_', '') for c in heat_data.columns],
    y=heat_data.index,
    colorscale='Blues',
    text=heat_data.values.astype(int),
    texttemplate='%{text} car.',
    hovertemplate='Domaine: <b>%{y}</b><br>Source: <b>%{x}</b><br>Moy: <b>%{z:.0f}</b> chars<extra></extra>'
))

fig.update_layout(
    title='🔥 Longueur moyenne des textes (caractères) par domaine et type de source',
    xaxis_title='Type de source',
    yaxis_title='Domaine',
    height=420,
    paper_bgcolor='white'
)
fig.show()

---
## 🔎 11. Explorateur Interactif de Textes

In [14]:
domain_sel  = widgets.Dropdown(
    options=['Tous'] + sorted(df['domain'].unique().tolist()),
    description='Domaine :',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)
source_sel  = widgets.Dropdown(
    options=['Tous'] + sorted(df['source_type'].unique().tolist()),
    description='Source :',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)
search_box  = widgets.Text(
    placeholder='Rechercher un mot...',
    description='Mot-clé :',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)
n_results   = widgets.IntSlider(
    value=10, min=5, max=50, step=5,
    description='Nb résultats :',
    style={'description_width': 'initial'}
)
btn2        = widgets.Button(description='🔍 Filtrer', button_style='primary')
output2     = widgets.Output()

def search_texts(b):
    with output2:
        output2.clear_output(wait=True)
        filtered = df.copy()
        if domain_sel.value != 'Tous':
            filtered = filtered[filtered['domain'] == domain_sel.value]
        if source_sel.value != 'Tous':
            filtered = filtered[filtered['source_type'] == source_sel.value]
        if search_box.value.strip():
            filtered = filtered[filtered['text'].str.contains(
                search_box.value.strip(), case=False, na=False
            )]
        sample = filtered.sample(min(n_results.value, len(filtered)))
        cols = ['id', 'text', 'domain', 'source_type', 'channel_name', 'text_length']
        cols = [c for c in cols if c in sample.columns]
        print(f'✅ {len(filtered):,} textes trouvés — affichage de {len(sample)}')
        display(sample[cols].style
                .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap',
                                   'max-width': '500px'})
                .background_gradient(subset=['text_length'], cmap='Blues')
        )

btn2.on_click(search_texts)

display(HTML('<h4>🔎 Explorateur interactif de textes</h4>'))
display(widgets.HBox([domain_sel, source_sel, search_box, n_results, btn2]))
display(output2)
search_texts(None)

Output()

---
## 📊 12. Résumé Statistique Complet

In [15]:
summary_rows = []

for domain in sorted(df['domain'].unique()):
    sub = df[df['domain'] == domain]
    summary_rows.append({
        'Domaine'          : domain,
        'Nb textes'        : len(sub),
        '% total'          : f"{len(sub)/total*100:.1f}%",
        'Lg moy (chars)'   : f"{sub['text_length'].mean():.0f}",
        'Lg med (chars)'   : f"{sub['text_length'].median():.0f}",
        'Mots moy'         : f"{sub['word_count'].mean():.1f}",
        'Commentaires'     : (sub['source_type'] == 'youtube_comment').sum(),
        'Transcriptions'   : (sub['source_type'] == 'youtube_transcript').sum(),
        'Arabe %'          : f"{(sub['script'] == 'Arabe').mean()*100:.1f}%",
        'Latin %'          : f"{(sub['script'] == 'Latin').mean()*100:.1f}%",
        'Mixte %'          : f"{(sub['script'] == 'Mixte').mean()*100:.1f}%",
        'Arabizi %'        : f"{(sub['script'] == 'Arabizi').mean()*100:.1f}%",
    })

summary_df = pd.DataFrame(summary_rows)

display(HTML('<h4>📊 Tableau de synthèse statistique complet</h4>'))
display(
    summary_df.style
    .background_gradient(subset=['Nb textes'], cmap='Blues')
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([{'selector': 'th', 'props': [
        ('background-color', '#003366'),
        ('color', 'white'),
        ('font-weight', 'bold'),
        ('text-align', 'center')
    ]}])
)

,Domaine,Nb textes,% total,Lg moy (chars),Lg med (chars),Mots moy,Commentaires,Transcriptions,Arabe %,Latin %,Mixte %,Arabizi %
0,business,19141,10.9%,76,48,13.8,8760,0,46.3%,22.2%,8.3%,23.0%
1,entertainment,35433,20.1%,49,29,8.0,35433,0,58.7%,15.9%,4.4%,17.7%
2,food,39019,22.2%,66,46,11.6,39019,0,81.8%,5.7%,5.1%,5.0%
3,medical,16299,9.3%,64,39,11.5,3007,0,61.8%,20.5%,6.2%,11.5%
4,miscill,4434,2.5%,92,56,16.2,4434,0,86.2%,4.3%,4.8%,4.1%
5,politics,32641,18.6%,107,59,18.7,8552,0,80.5%,7.0%,6.6%,5.4%
6,religion,16438,9.3%,124,62,22.2,16438,0,91.1%,2.3%,4.4%,1.6%
7,sports,12452,7.1%,70,40,12.6,8074,0,48.1%,25.6%,4.2%,21.5%


---
## 💾 13. Export des Statistiques

In [16]:
# Export CSV du résumé
summary_df.to_csv('stats_summary.csv', index=False, encoding='utf-8')
print('✅ stats_summary.csv exporté')

# Export rapport texte
with open('stats_report.txt', 'w', encoding='utf-8') as f:
    f.write('=' * 60 + '\n')
    f.write('RAPPORT STATISTIQUE — CORPUS DARIJA MAROCAINE\n')
    f.write('=' * 60 + '\n\n')
    f.write(f'Total textes          : {total:,}\n')
    f.write(f'Longueur moy (chars)  : {df["text_length"].mean():.1f}\n')
    f.write(f'Longueur med (chars)  : {df["text_length"].median():.1f}\n')
    f.write(f'Nb mots moyen         : {df["word_count"].mean():.1f}\n\n')
    f.write('--- Par domaine ---\n')
    for _, row in domain_counts.iterrows():
        f.write(f"  {row['Domaine']:20s} : {row['Nombre']:6,} ({row['Pourcentage']}%)\n")
    f.write('\n--- Par script ---\n')
    for _, row in script_counts.iterrows():
        pct = row['Nombre'] / total * 100
        f.write(f"  {row['Script']:15s} : {row['Nombre']:6,} ({pct:.1f}%)\n")

print('✅ stats_report.txt exporté')
print('\n📁 Fichiers générés :')
print('   - stats_summary.csv')
print('   - stats_report.txt')

✅ stats_summary.csv exporté
✅ stats_report.txt exporté

📁 Fichiers générés :
   - stats_summary.csv
   - stats_report.txt
